In [2]:
import altair as alt
import datetime as dt
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from datetime import datetime
from scipy.stats import norm
from toolz.curried import *

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [31]:
timestamps = (
    int(dt.datetime(2026, 2, 6, 0, 0).timestamp()),
    int(dt.datetime(2026, 2, 6, 12, 0).timestamp()),
)

resolution = "5m"

dfs = {}

In [32]:
BASE_URL = "https://thalex.com/api/v2/public"
endpoint = "mark_price_historical_data"
url = f"{BASE_URL}/{endpoint}"

params = lambda instrument_name: {
    "from": timestamps[0],
    "to": timestamps[1],
    "resolution": resolution,
    "instrument_name": instrument_name,
}

COLUMNS = [
    "ts",
    "mark_price_open",
    "mark_price_high",
    "mark_price_low",
    "mark_price_close",
    "iv_open",
    "iv_high",
    "iv_low",
    "iv_close",
    "tob",
]


instrument_names = [
    f"BTC-13FEB26-70000-C",
]

dfs["option"] = (
    pipe(
        {name: requests.get(url, params=params(name)) for name in instrument_names},
        valmap(requests.Response.json),
        valmap(get_in(["result", "mark"])),
        valmap(curry(pd.DataFrame, columns=COLUMNS)),
        pd.concat,
    )
    .droplevel(1)
    .reset_index(names=["instrument_name"])
)

dfs["option"]

,instrument_name,ts,mark_price_open,mark_price_high,mark_price_low,mark_price_close,iv_open,iv_high,iv_low,iv_close,tob
0,BTC-13FEB26-70000-C,1.770332e+09,1025.706789,1223.748589,1006.728790,1215.721142,0.846713,0.849064,0.832998,0.838506,None
1,BTC-13FEB26-70000-C,1.770333e+09,1214.614431,1256.272960,1075.820713,1133.776106,0.838752,0.840786,0.825102,0.825301,None
2,BTC-13FEB26-70000-C,1.770333e+09,1132.763965,1169.867608,1114.219093,1115.182021,0.825331,0.825363,0.818423,0.821209,None
3,BTC-13FEB26-70000-C,1.770333e+09,1113.561412,1144.678579,1028.662336,1028.662336,0.821251,0.821469,0.808780,0.809422,None
4,BTC-13FEB26-70000-C,1.770334e+09,1026.530676,1034.280607,956.297763,956.297763,0.809469,0.816515,0.809469,0.811659,None
...,...,...,...,...,...,...,...,...,...,...,...
139,BTC-13FEB26-70000-C,1.770374e+09,996.231158,1017.132634,968.389580,972.715394,0.675142,0.676017,0.672864,0.674845,None
140,BTC-13FEB26-70000-C,1.770374e+09,972.396382,1002.771134,968.236695,990.669374,0.674856,0.676952,0.672345,0.672345,None
141,BTC-13FEB26-70000-C,1.770375e+09,990.961873,999.091731,966.296726,973.413203,0.672346,0.675233,0.672342,0.675233,None
142,BTC-13FEB26-70000-C,1.770375e+09,972.464474,981.619142,928.095297,969.420808,0.675299,0.678012,0.674644,0.675027,None


In [33]:
endpoint = "instrument"
url = f"{BASE_URL}/{endpoint}"

params = lambda instrument_name: {
    "instrument_name": instrument_name,
}

instrument_data = pipe(
    {name: requests.get(url, params=params(name)) for name in instrument_names},
    valmap(requests.Response.json),
    valmap(get("result")),
    valmap(
        keyfilter(
            lambda k: k in ["expiration_timestamp", "option_type", "strike_price"]
        )
    ),
)

instrument_data

{'BTC-13FEB26-70000-C': {'option_type': 'call',
  'expiration_timestamp': 1770969600,
  'strike_price': 70000.0}}

In [34]:
BASE_URL = "https://thalex.com/api/v2/public"
endpoint = "index_price_historical_data"
url = f"{BASE_URL}/{endpoint}"

params = {
    "index_name": "BTCUSD",
    "resolution": resolution,
    "from": timestamps[0],
    "to": timestamps[1],
}

COLS = [
    "ts",
    "index_price_open",
    "index_price_high",
    "index_price_low",
    "index_price_close",
]


dfs["index"] = pipe(
    requests.get(url, params=params),
    requests.Response.json,
    get_in(["result", "index"]),
    curry(
        pd.DataFrame,
        columns=COLS,
    ),
)

dfs["index"]

,ts,index_price_open,index_price_high,index_price_low,index_price_close
0,1.770332e+09,63795.643500,64640.785167,63718.163500,64611.188333
1,1.770333e+09,64605.080167,64751.246667,64227.218333,64469.708167
2,1.770333e+09,64465.666667,64662.125167,64451.808333,64451.808333
3,1.770333e+09,64445.601667,64594.198333,64257.373333,64257.373333
4,1.770334e+09,64248.670000,64271.248333,63923.291667,63923.291667
...,...,...,...,...,...
139,1.770374e+09,65969.645000,66067.503333,65871.871667,65891.496667
140,1.770374e+09,65890.303333,66010.980000,65874.710000,65993.001667
141,1.770375e+09,65994.241000,66011.898333,65888.588500,65903.328333
142,1.770375e+09,65899.245000,65921.861667,65726.816667,65884.456667


In [35]:
def calc_delta(spot, strike, time_to_expiry, iv, option_type="call"):
    tau = time_to_expiry / (365.25 * 24 * 60 * 60)
    d1 = (np.log(spot / strike) + 0.5 * iv**2 * tau) / (iv * np.sqrt(tau))
    if option_type == "call":
        return norm.cdf(d1)
    else:
        return norm.cdf(d1) - 1


dfs["delta"] = pd.merge(dfs["index"], dfs["option"], on="ts").assign(
    date_time=lambda df: pd.to_datetime(df["ts"], unit="s"),
    expiration=lambda df: df["instrument_name"].map(
        lambda name: get_in([name, "expiration_timestamp"], instrument_data)
    ),
    type=lambda df: df["instrument_name"].map(
        lambda name: get_in([name, "option_type"], instrument_data)
    ),
    strike=lambda df: df["instrument_name"]
    .map(lambda name: get_in([name, "strike_price"], instrument_data))
    .astype(int),
    tte=lambda df: df["expiration"] - df["ts"],
    delta=lambda df: df.apply(
        lambda row: calc_delta(
            row["index_price_close"],
            row["strike"],
            row["tte"],
            row["iv_close"],
            row["type"],
        ),
        axis=1,
    ),
)

dfs["delta"]

,ts,index_price_open,index_price_high,index_price_low,index_price_close,instrument_name,mark_price_open,mark_price_high,mark_price_low,mark_price_close,...,iv_high,iv_low,iv_close,tob,date_time,expiration,type,strike,tte,delta
0,1.770332e+09,63795.643500,64640.785167,63718.163500,64611.188333,BTC-13FEB26-70000-C,1025.706789,1223.748589,1006.728790,1215.721142,...,0.849064,0.832998,0.838506,None,2026-02-05 23:00:00,1770969600,call,70000,637200.0,0.270019
1,1.770333e+09,64605.080167,64751.246667,64227.218333,64469.708167,BTC-13FEB26-70000-C,1214.614431,1256.272960,1075.820713,1133.776106,...,0.840786,0.825102,0.825301,None,2026-02-05 23:05:00,1770969600,call,70000,636900.0,0.260008
2,1.770333e+09,64465.666667,64662.125167,64451.808333,64451.808333,BTC-13FEB26-70000-C,1132.763965,1169.867608,1114.219093,1115.182021,...,0.825363,0.818423,0.821209,None,2026-02-05 23:10:00,1770969600,call,70000,636600.0,0.257952
3,1.770333e+09,64445.601667,64594.198333,64257.373333,64257.373333,BTC-13FEB26-70000-C,1113.561412,1144.678579,1028.662336,1028.662336,...,0.821469,0.808780,0.809422,None,2026-02-05 23:15:00,1770969600,call,70000,636300.0,0.245951
4,1.770334e+09,64248.670000,64271.248333,63923.291667,63923.291667,BTC-13FEB26-70000-C,1026.530676,1034.280607,956.297763,956.297763,...,0.816515,0.809469,0.811659,None,2026-02-05 23:20:00,1770969600,call,70000,636000.0,0.232541
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,1.770374e+09,65969.645000,66067.503333,65871.871667,65891.496667,BTC-13FEB26-70000-C,996.231158,1017.132634,968.389580,972.715394,...,0.676017,0.672864,0.674845,None,2026-02-06 10:35:00,1770969600,call,70000,595500.0,0.272218
140,1.770374e+09,65890.303333,66010.980000,65874.710000,65993.001667,BTC-13FEB26-70000-C,972.396382,1002.771134,968.236695,990.669374,...,0.676952,0.672345,0.672345,None,2026-02-06 10:40:00,1770969600,call,70000,595200.0,0.276851
141,1.770375e+09,65994.241000,66011.898333,65888.588500,65903.328333,BTC-13FEB26-70000-C,990.961873,999.091731,966.296726,973.413203,...,0.675233,0.672342,0.675233,None,2026-02-06 10:45:00,1770969600,call,70000,594900.0,0.272878
142,1.770375e+09,65899.245000,65921.861667,65726.816667,65884.456667,BTC-13FEB26-70000-C,972.464474,981.619142,928.095297,969.420808,...,0.678012,0.674644,0.675027,None,2026-02-06 10:50:00,1770969600,call,70000,594600.0,0.271722


In [36]:
dfs["delta"].to_csv("delta.csv")

In [37]:
dfs["hedge"] = dfs["delta"].assign(
    p=lambda df: df["index_price_close"],
    p_prev=lambda df: df["index_price_open"],
    p_chg=lambda df: df["index_price_close"] - df["index_price_open"],
    delta_prev=lambda df: df.groupby("instrument_name")["delta"].shift(1).fillna(0),
    delta_change=lambda df: df["delta"] - df["delta_prev"],
    q=lambda df: -df["delta_change"],
    q_buy=lambda df: np.where(df["q"] > 0, df["q"], 0.0),
    q_sell=lambda df: np.where(df["q"] < 0, -df["q"], 0.0),
    v_buy=lambda df: df["q_buy"] * df["p"],
    v_sell=lambda df: df["q_sell"] * df["p"],
    q_buy_cum=lambda df: df.groupby("instrument_name")["q_buy"].cumsum(),
    v_buy_cum=lambda df: df.groupby("instrument_name")["v_buy"].cumsum(),
    q_sell_cum=lambda df: df.groupby("instrument_name")["q_sell"].cumsum(),
    v_sell_cum=lambda df: df.groupby("instrument_name")["v_sell"].cumsum(),
    p_buy=lambda df: ((df["v_buy_cum"] / df["q_buy_cum"]).where(df["q_buy_cum"] > 0))
    .groupby(df["instrument_name"])
    .ffill(),
    hedge_position=lambda df: -df["delta_prev"],
    realized=lambda df: (df["v_sell_cum"] - df["q_sell_cum"] * df["p_buy"]).fillna(0),
    unrealized=lambda df: df["hedge_position"]
    * (df["p"] - df["p_buy"].fillna(df["p"])),
    hedge_pnl_cumul=lambda df: df["realized"] + df["unrealized"],
    # hedge_pnl=lambda df: df["hedge_position"] * df["p_chg"],
    # hedge_pnl_cumul=lambda df: df["hedge_pnl"].cumsum(),
    initial_mark=lambda df: df.groupby("instrument_name")["mark_price_open"].transform(
        "first"
    ),
    option_pnl_cumul=lambda df: df["mark_price_close"] - df["initial_mark"],
    pnl=lambda df: df["hedge_pnl_cumul"] - df["option_pnl_cumul"],
    tx_costs=lambda df: (df["v_buy"] + df["v_sell"]) * 0.0001,
    tx_costs_cumul=lambda df: df["tx_costs"].cumsum(),
    hedge_pnl_tx_cumul=lambda df: df["hedge_pnl_cumul"] - df["tx_costs_cumul"],
)

dfs["hedge"]

,ts,index_price_open,index_price_high,index_price_low,index_price_close,instrument_name,mark_price_open,mark_price_high,mark_price_low,mark_price_close,...,hedge_position,realized,unrealized,hedge_pnl_cumul,initial_mark,option_pnl_cumul,pnl,tx_costs,tx_costs_cumul,hedge_pnl_tx_cumul
0,1.770332e+09,63795.643500,64640.785167,63718.163500,64611.188333,BTC-13FEB26-70000-C,1025.706789,1223.748589,1006.728790,1215.721142,...,-0.000000,0.000000,-0.000000e+00,0.000000,1025.706789,190.014353,-190.014353,1.744628,1.744628,-1.744628
1,1.770333e+09,64605.080167,64751.246667,64227.218333,64469.708167,BTC-13FEB26-70000-C,1214.614431,1256.272960,1075.820713,1133.776106,...,-0.270019,38.202403,-1.964650e-12,38.202403,1025.706789,108.069317,-69.866913,0.064546,1.809174,36.393229
2,1.770333e+09,64465.666667,64662.125167,64451.808333,64451.808333,BTC-13FEB26-70000-C,1132.763965,1169.867608,1114.219093,1115.182021,...,-0.260008,39.025615,3.861405e+00,42.887020,1025.706789,89.475232,-46.588212,0.013247,1.822421,41.064599
3,1.770333e+09,64445.601667,64594.198333,64257.373333,64257.373333,BTC-13FEB26-70000-C,1113.561412,1144.678579,1028.662336,1028.662336,...,-0.257952,67.204293,2.706647e+01,94.270765,1025.706789,2.955547,91.315218,0.077119,1.899540,92.371226
4,1.770334e+09,64248.670000,64271.248333,63923.291667,63923.291667,BTC-13FEB26-70000-C,1026.530676,1034.280607,956.297763,956.297763,...,-0.245951,109.618019,6.934173e+01,178.959746,1025.706789,-69.409026,248.368772,0.085719,1.985259,176.974488
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,1.770374e+09,65969.645000,66067.503333,65871.871667,65891.496667,BTC-13FEB26-70000-C,996.231158,1017.132634,968.389580,972.715394,...,-0.276993,255.135379,-4.192435e+02,-164.108129,1025.706789,-52.991396,-111.116734,0.031461,10.182266,-174.290395
140,1.770374e+09,65890.303333,66010.980000,65874.710000,65993.001667,BTC-13FEB26-70000-C,972.396382,1002.771134,968.236695,990.669374,...,-0.272218,262.617225,-4.396482e+02,-177.030961,1025.706789,-35.037416,-141.993546,0.030572,10.212837,-187.243799
141,1.770375e+09,65994.241000,66011.898333,65888.588500,65903.328333,BTC-13FEB26-70000-C,990.961873,999.091731,966.296726,973.413203,...,-0.276851,254.038889,-4.197491e+02,-165.710201,1025.706789,-52.293587,-113.416614,0.026183,10.239020,-175.949220
142,1.770375e+09,65899.245000,65921.861667,65726.816667,65884.456667,BTC-13FEB26-70000-C,972.464474,981.619142,928.095297,969.420808,...,-0.272878,251.594749,-4.078585e+02,-156.263701,1025.706789,-56.285981,-99.977720,0.007611,10.246631,-166.510332


In [38]:
df = pd.concat(
    [
        dfs["hedge"].assign(
            category="Replicator", value=lambda df: -df["hedge_pnl_cumul"]
        ),
        dfs["hedge"].assign(category="Option", value=lambda df: df["option_pnl_cumul"]),
        dfs["hedge"].assign(
            category="repl_pnl_after_fees", value=lambda df: -df["hedge_pnl_tx_cumul"]
        ),
    ],
)

df

,ts,index_price_open,index_price_high,index_price_low,index_price_close,instrument_name,mark_price_open,mark_price_high,mark_price_low,mark_price_close,...,unrealized,hedge_pnl_cumul,initial_mark,option_pnl_cumul,pnl,tx_costs,tx_costs_cumul,hedge_pnl_tx_cumul,category,value
0,1.770332e+09,63795.643500,64640.785167,63718.163500,64611.188333,BTC-13FEB26-70000-C,1025.706789,1223.748589,1006.728790,1215.721142,...,-0.000000e+00,0.000000,1025.706789,190.014353,-190.014353,1.744628,1.744628,-1.744628,Replicator,-0.000000
1,1.770333e+09,64605.080167,64751.246667,64227.218333,64469.708167,BTC-13FEB26-70000-C,1214.614431,1256.272960,1075.820713,1133.776106,...,-1.964650e-12,38.202403,1025.706789,108.069317,-69.866913,0.064546,1.809174,36.393229,Replicator,-38.202403
2,1.770333e+09,64465.666667,64662.125167,64451.808333,64451.808333,BTC-13FEB26-70000-C,1132.763965,1169.867608,1114.219093,1115.182021,...,3.861405e+00,42.887020,1025.706789,89.475232,-46.588212,0.013247,1.822421,41.064599,Replicator,-42.887020
3,1.770333e+09,64445.601667,64594.198333,64257.373333,64257.373333,BTC-13FEB26-70000-C,1113.561412,1144.678579,1028.662336,1028.662336,...,2.706647e+01,94.270765,1025.706789,2.955547,91.315218,0.077119,1.899540,92.371226,Replicator,-94.270765
4,1.770334e+09,64248.670000,64271.248333,63923.291667,63923.291667,BTC-13FEB26-70000-C,1026.530676,1034.280607,956.297763,956.297763,...,6.934173e+01,178.959746,1025.706789,-69.409026,248.368772,0.085719,1.985259,176.974488,Replicator,-178.959746
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,1.770374e+09,65969.645000,66067.503333,65871.871667,65891.496667,BTC-13FEB26-70000-C,996.231158,1017.132634,968.389580,972.715394,...,-4.192435e+02,-164.108129,1025.706789,-52.991396,-111.116734,0.031461,10.182266,-174.290395,repl_pnl_after_fees,174.290395
140,1.770374e+09,65890.303333,66010.980000,65874.710000,65993.001667,BTC-13FEB26-70000-C,972.396382,1002.771134,968.236695,990.669374,...,-4.396482e+02,-177.030961,1025.706789,-35.037416,-141.993546,0.030572,10.212837,-187.243799,repl_pnl_after_fees,187.243799
141,1.770375e+09,65994.241000,66011.898333,65888.588500,65903.328333,BTC-13FEB26-70000-C,990.961873,999.091731,966.296726,973.413203,...,-4.197491e+02,-165.710201,1025.706789,-52.293587,-113.416614,0.026183,10.239020,-175.949220,repl_pnl_after_fees,175.949220
142,1.770375e+09,65899.245000,65921.861667,65726.816667,65884.456667,BTC-13FEB26-70000-C,972.464474,981.619142,928.095297,969.420808,...,-4.078585e+02,-156.263701,1025.706789,-56.285981,-99.977720,0.007611,10.246631,-166.510332,repl_pnl_after_fees,166.510332


In [39]:
df["v_buy_cum"] + df["v_sell_cum"]

0       17446.280458
1       18091.741561
2       18224.208221
3       18995.396518
4       19852.585205
           ...      
139    101822.655311
140    102128.371344
141    102390.197111
142    102466.308424
143    102510.010653
Length: 432, dtype: float64

In [40]:
a = (
    alt.Chart(
        df,
        width=1600,
        height=900,
        title=alt.Title(
            f"Delta of {instrument_names[0]}",
            color="white",
            fontSize=22,
            offset=60,
            anchor="middle",
        ),
    )
    .mark_line(interpolate="basis", strokeWidth=1, opacity=0.9)
    .encode(
        x=alt.X("date_time", title=None),
        y="value",
        color=alt.Color(
            "category",
            scale=alt.Scale(scheme="tableau20"),
        ),
    )
)


chart = (
    a.configure(
        background="black",
        padding={"right": 40, "left": 40, "top": 40, "bottom": 40},
    )
    .configure_legend(
        labelColor="white",
        labelFontSize=13,
        # padding=40,
        titleColor="white",
        titleFontSize=16,
        titlePadding=8,
        rowPadding=6,
    )
    .configure_axis(
        domain=False,
        grid=False,
        labelColor="white",
        labelFontSize=12,
        labelPadding=20,
        tickCount=5,
        ticks=False,
        titleColor="white",
        titleFontSize=16,
        titlePadding=40,
    )
    .configure_axisX(grid=False, tickCount=10)
    .configure_view(stroke=None)
)


chart

alt.Chart(...)

In [41]:
marks = (
    alt.Chart(
        df,
        width=1600,
        height=450,
        title=alt.Title(
            f"{instrument_names[0]} - Mark Prices by Maturity",
            color="white",
            fontSize=22,
            offset=60,
            anchor="middle",
        ),
    )
    .mark_line(interpolate="basis", strokeWidth=3, opacity=0.9)
    .encode(
        x=alt.X(
            "date_time",
            title=None,
            axis=alt.Axis(labelColor="black", titleColor="black"),
        ),
        y="pnl",
        color=alt.Color(
            "instrument_name",
            scale=alt.Scale(scheme="tableau20"),
            sort=alt.SortField("strike"),
        ),
    )
)

deltas = (
    alt.Chart(
        df,
        width=1600,
        height=450,
    )
    .mark_line(
        interpolate="natural",
        strokeWidth=3,
        opacity=0.9,
        strokeDash=[4, 2],
    )
    .encode(
        x=alt.X("date_time", title=None),
        y=alt.Y("delta"),
        color=alt.Color(
            "instrument_name",
            scale=alt.Scale(scheme="tableau20"),
            sort=alt.SortField("strike"),
        ),
    )
)

idx = (
    alt.Chart(
        df,
        width=1600,
        height=450,
    )
    .mark_line(color="mistyrose", strokeWidth=3, interpolate="basis", opacity=0.9)
    .encode(x="date_time", y=alt.Y("index_price_close", scale=alt.Scale(zero=False)))
)

chart = (
    alt.vconcat((marks).resolve_scale(y="independent"), idx, spacing=20)
    .configure(
        background="black",
        padding={"right": 40, "left": 40, "top": 40, "bottom": 40},
    )
    .configure_legend(
        labelColor="white",
        labelFontSize=13,
        # padding=40,
        titleColor="white",
        titleFontSize=16,
        titlePadding=8,
        rowPadding=6,
    )
    .configure_axis(
        domain=False,
        grid=False,
        labelColor="white",
        labelFontSize=12,
        labelPadding=20,
        tickCount=5,
        ticks=False,
        titleColor="white",
        titleFontSize=16,
        titlePadding=40,
    )
    .configure_axisX(grid=False, tickCount=10)
    .configure_view(stroke=None)
)

chart

alt.VConcatChart(...)

In [42]:
a = (
    alt.Chart(
        df,
        width=1600,
        height=600,
        title=alt.Title(
            f"Delta of {instrument_names[0]}",
            color="white",
            fontSize=22,
            offset=60,
            anchor="middle",
        ),
    )
    .mark_line(interpolate="basis", strokeWidth=3, opacity=0.9)
    .encode(
        x=alt.X("date_time", title=None),
        y="value",
        color=alt.Color(
            "category",
            scale=alt.Scale(scheme="tableau20"),
            sort=alt.SortField("strike"),
        ),
    )
)

b = (
    alt.Chart(
        df,
        width=1600,
        height=600,
    )
    # Drop stroke to avoid dark edge at x=0 on black background
    .mark_bar(opacity=0.9, stroke=None).encode(
        x=alt.X("date_time", title=None),
        y=alt.Y("hedge_position", title="hedge position"),
    )
)

c = (
    alt.Chart(
        df,
        width=1600,
        height=600,
    )
    .mark_line(color="mistyrose", strokeWidth=3, interpolate="basis", opacity=0.9)
    .encode(x="date_time", y=alt.Y("index_price_close", scale=alt.Scale(zero=False)))
)

chart = (
    alt.vconcat(a, c, spacing=50)
    .configure(
        background="black",
        padding={"right": 40, "left": 40, "top": 40, "bottom": 40},
    )
    .configure_legend(
        labelColor="white",
        labelFontSize=13,
        # padding=40,
        titleColor="white",
        titleFontSize=16,
        titlePadding=8,
        rowPadding=6,
    )
    .configure_axis(
        domain=False,
        grid=False,
        labelColor="white",
        labelFontSize=12,
        labelPadding=20,
        tickCount=5,
        ticks=False,
        titleColor="white",
        titleFontSize=16,
        titlePadding=40,
    )
    .configure_axisX(grid=False, tickCount=10)
    .configure_view(stroke=None)
)


chart

alt.VConcatChart(...)

In [43]:
chart.save("fastest_horse.png", scale_factor=4)

In [29]:
a = (
    alt.Chart(
        df,
        width=1600,
        height=600,
        title=alt.Title(
            f"Delta of {instrument_names[0]}",
            color="white",
            fontSize=22,
            offset=60,
            anchor="middle",
        ),
    )
    .mark_line(interpolate="basis", strokeWidth=3, opacity=0.9)
    .encode(
        x=alt.X("date_time", title=None),
        y="value",
        color=alt.Color(
            "category",
            scale=alt.Scale(scheme="tableau20"),
            sort=alt.SortField("strike"),
        ),
    )
)

b = (
    alt.Chart(
        df,
        width=1600,
        height=600,
    )
    # Drop stroke to avoid dark edge at x=0 on black background
    .mark_bar(opacity=0.9, stroke=None).encode(
        x=alt.X("date_time", title=None),
        y=alt.Y("hedge_position", title="hedge position"),
    )
)

c = (
    alt.Chart(
        df,
        width=1600,
        height=600,
    )
    .mark_line(color="mistyrose", strokeWidth=3, interpolate="basis", opacity=0.9)
    .encode(x="date_time", y=alt.Y("index_price_close", scale=alt.Scale(zero=False)))
)

chart = (
    alt.vconcat(a, c, spacing=50)
    .configure(
        background="black",
        padding={"right": 40, "left": 40, "top": 40, "bottom": 40},
    )
    .configure_legend(
        labelColor="white",
        labelFontSize=13,
        # padding=40,
        titleColor="white",
        titleFontSize=16,
        titlePadding=8,
        rowPadding=6,
    )
    .configure_axis(
        domain=False,
        grid=False,
        labelColor="white",
        labelFontSize=12,
        labelPadding=20,
        tickCount=5,
        ticks=False,
        titleColor="white",
        titleFontSize=16,
        titlePadding=40,
    )
    .configure_axisX(grid=False, tickCount=10)
    .configure_view(stroke=None)
)


chart

alt.VConcatChart(...)

In [17]:
chart.save("hedging_cost.png", scale_factor=3)

/opt/miniconda3/lib/python3.13/site-packages/altair/utils/core.py:264: UserWarning: I don't know how to infer vegalite type from 'empty'.  Defaulting to nominal.
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/altair/utils/core.py:264: UserWarning: I don't know how to infer vegalite type from 'empty'.  Defaulting to nominal.
  warnings.warn(
